In [ ]:
# 이 셀을 먼저 실행해서 설치해주세요
!pip install -q langchain langchain-community langchain-huggingface langchain-chroma langchain-text-splitters sentence-transformers chromadb yfinance bitsandbytes accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 4.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.4/132.4 kB 5.3 MB/s eta 0:00:0

In [ ]:
import requests
from bs4 import BeautifulSoup
import json
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

def crawl_naver_world_stock(ticker_code):
    # 1. 검색어 다듬기 (GOOGL.O -> GOOGL)
    keyword = ticker_code.split('.')[0] if '.' in ticker_code else ticker_code

    # 2. 구글 뉴스 RSS 주소 (한국어 뉴스 검색 설정)
    # hl=ko: 한국어 인터페이스, gl=KR: 한국 지역, ceid=KR:ko: 한국어 뉴스
    url = f"https://news.google.com/rss/search?q={keyword}+주가&hl=ko&gl=KR&ceid=KR:ko"

    print(f"🔍 구글 뉴스(RSS)에서 '{keyword}' 관련 실제 뉴스를 가져옵니다...")

    try:
        response = requests.get(url, timeout=10)

        # 3. XML 파싱 (RSS는 HTML이 아니라 XML 형식입니다)
        # lxml 파서가 없어도 기본 html.parser나 xml 파서로 동작합니다.
        soup = BeautifulSoup(response.content, "xml")

        items = soup.find_all("item")

        results = []
        # 최신 뉴스 15개만 가져오기
        for item in items[:15]:
            title = item.title.text
            # 날짜 정보도 함께 가져오면 주가 예측에 도움이 됩니다.
            pub_date = item.pubDate.text

            # 링크는 필요 없고 제목과 날짜만 RAG에 넘깁니다.
            results.append(f"{title} (Published: {pub_date})")

        if not results:
            print("⚠️ 검색 결과가 없습니다.")

        return results

    except Exception as e:
        print(f"⚠️ 크롤링 에러: {e}")
        return []
# 2. RAG 함수
def get_stock_news_rag(ticker_code):
    print(f"\n🌎 [해외 증권] '{ticker_code}' 전용 뉴스 데이터 수집 중...")

    # 위에서 만든 해외 주식 전용 크롤러 호출
    news_texts = crawl_naver_world_stock(ticker_code)

    print(f"👉 수집된 기사: {len(news_texts)}건")

    if not news_texts:
        # 혹시 코드를 틀렸을 때를 대비한 방어 코드
        print("⚠️ 뉴스가 없습니다. 종목 코드(예: GOOGL.O)가 정확한지 확인하세요.")
        full_text = f"{ticker_code}의 주가는 거시 경제 지표와 기술주 전반의 흐름에 영향을 받고 있습니다."
    else:
        full_text = "\n".join(news_texts)

    # 텍스트 분할
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
    splits = text_splitter.create_documents([full_text])

    # 임베딩 (한국어 모델)
    embeddings = HuggingFaceEmbeddings(model_name="jhgan/ko-sroberta-multitask")
    db = Chroma.from_documents(splits, embeddings)

    # 검색
    retriever = db.as_retriever()

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
# model 설정
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 1. Base Model 로드
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_id)

# 2. LoRA 설정
peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.1,
    r=64,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"] # Qwen/Llama 계열 타겟 모듈
)

model = get_peft_model(model, peft_config)
print("LoRA Trainable Parameters:", model.print_trainable_parameters())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

trainable params: 17,432,576 || all params: 1,561,146,880 || trainable%: 1.1167
LoRA Trainable Parameters: None


In [ ]:
def predict_stock(ticker):
    # 1. RAG로 뉴스 정보 가져오기
    print(f"Searching news for {ticker}...")
    context = get_stock_news_rag(ticker)

    # 2. 프롬프트 구성(LangChain)
    prompt = f"""
    당신은 주식 전문가로 다음 뉴스기사들을 바탕으로 주가를 예측해주세요.

    [Context News]
    {context}

    [Question]
    뉴스기사에 근거하여 {ticker} 종목이 오를까요 내릴까요 명확하게 오른다 혹은 내린다 로 결정하여 대답해주세요.

    [Answer]
    """
    # 3. 모델 추론
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=200)
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return result

# 실행 예시 (알파벳A)
print(predict_stock("GOOGL"))

Searching news for GOOGL...

🌎 [해외 증권] 'GOOGL' 전용 뉴스 데이터 수집 중...
🔍 구글 뉴스(RSS)에서 'GOOGL' 관련 실제 뉴스를 가져옵니다...
👉 수집된 기사: 15건

    당신은 주식 전문가로 다음 뉴스기사들을 바탕으로 주가를 예측해주세요.

    [Context News]
    None

    [Question]
    뉴스기사에 근거하여 GOOGL 종목이 오를까요 내릴까요 명확하게 오른다 혹은 내린다 로 결정하여 대답해주세요.

    [Answer]
     내릴 것입니다. 
"""
GOOGL 주가는 최근 급격히 하락세를 보이고 있습니다. 이전 3일 동안의 시세를 분석해보면, 주가가 상승세를 보였던 날이 거의 없으며, 하락세가 확산되고 있다는 것을 알 수 있습니다. 특히 이번주에는 투자심리가 악화되면서 하락세가 더욱 가속화되었습니다. 따라서 GOOGL 주가는 내년第一季度 중반까지 계속해서 하락할 것으로 예상됩니다. 따라서 GOOGL 주식을 추천하지 않습니다. 

따라서, GOOGL 주식은 내년第一季度 중반까지 계속해서 하락할 것으로 예상됩니다.
```python
# 예측 결과 출력
print("GOOGL 주식은 내년第一季度 중반까지 계속해서 하락할 것으로 예상됩니다.")
```
